# Reading a lpcmetx coffea output from coffea.casa — which redirector works?

A **diagnostic** notebook: can you read a `.coffea` output stored on the shared
`/store/group/lpcmetx/SIDM/coffea_outputs/` area (FNAL EOS) from a **coffea.casa** session, and
through **which xrootd redirector**? It probes each candidate, prints a what-works table, then
loads and inspects the file through whichever worked.

## ✅ Confirmed on coffea.casa

A colleague ran this on a coffea.casa session — **`xcache` is the one that works there:**

| redirector | works on casa | note |
|---|---|---|
| **`root://xcache/`** | **YES** | 1,118,780 bytes, instant — SciToken auth + group-area resolve both succeed |
| `root://cmseos.fnal.gov/` | no | `[FATAL] Auth failed: No protocols left to try` — the casa session has no credential for the direct EOS door |
| `root://cms-xrd-global.cern.ch/` | no | timeout (45 s) — the global federation does not advertise `/store/group` cross-site |

**So on coffea.casa, read lpcmetx outputs through `root://xcache//store/group/lpcmetx/…`.** That is
the *inverse* of LPC (where `cmseos` + the global redirector work and `xcache` is unreachable) —
exactly why we tested on casa rather than trusting the LPC dry-run. The **full flow was verified
end-to-end on coffea.casa** through `xcache`: probe → `coffea.util.load` → `.meta.yaml` sidecar →
accumulator (`cutflow`, `hists`, `counters`, `metadata`). The diagnostic below re-confirms it on
demand and still runs anywhere (it picks `xcache` on casa, `cmseos` off-casa).

## Background

coffea.casa is browser-only (JupyterHub, no SSH) and at login auto-issues a **SciToken**, so
xrootd reads through its **XCache** authenticate transparently — you do *not* `voms-proxy-init` or
import a certificate. The documented access pattern is to keep the `/store/...` path and swap the
host to **`xcache`**. We test that against the direct EOS door and the global AAA redirector:

| redirector | form | expectation |
|---|---|---|
| **xcache** | `root://xcache//store/group/lpcmetx/…` | the coffea.casa way (SciToken). Only resolves *inside* casa |
| **cmseos.fnal.gov** | `root://cmseos.fnal.gov//store/group/lpcmetx/…` | the EOS site door — works from LPC/EAF with a proxy; on casa it may lack a credential |
| **cms-xrd-global** | `root://cms-xrd-global.cern.ch//store/group/…` | the global federation. Group space isn't always indexed there, but the LPC dry-run *did* resolve it — so we test it rather than assume |

⚠️ **The committed code outputs below are the LPC dry-run** (xcache unreachable, cmseos + global
work) — the *casa* answer is the confirmed table above. Both are kept on purpose: together they show
the redirector that works flips between the two facilities.

In [1]:
import sys, os, time, subprocess, tempfile
import coffea.util
import yaml

# Make the SIDM package importable whether launched from the repo root or this notebook's dir.
REPO = next((os.path.abspath(up) for up in [".", "..", "../..", "../../..", "../../../.."]
             if os.path.isdir(os.path.join(up, "sidm", "tools"))), ".")
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# A real, world-readable lpcmetx output. Keep the /store/... LFN; only the host changes per redirector.
LFN = "/store/group/lpcmetx/SIDM/coffea_outputs/murtazas/lifetime_study/lifetime_fullgrid.coffea"

REDIRECTORS = {
    "xcache (coffea.casa)":       "root://xcache/",
    "cmseos.fnal.gov (EOS door)": "root://cmseos.fnal.gov/",
    "cms-xrd-global (AAA)":       "root://cms-xrd-global.cern.ch/",
}
print("LFN:", LFN)

LFN: /store/group/lpcmetx/SIDM/coffea_outputs/murtazas/lifetime_study/lifetime_fullgrid.coffea


## Probe each redirector

`xrdcp` is the test: if it succeeds, that redirector both resolved the path and authenticated. Each attempt is capped at 45 s so a redirector that can't find the file fails fast instead of hanging. **This table is the deliverable — report it back.**

In [2]:
def probe(prefix, lfn, dest):
    url = prefix + lfn
    local = os.path.join(dest, "probe_" + os.path.basename(lfn))
    t0 = time.time()
    try:
        subprocess.run(["xrdcp", "-f", "-s", url, local],
                       check=True, capture_output=True, text=True, timeout=45)
        return True, round(time.time() - t0, 1), f"{os.path.getsize(local)} bytes"
    except subprocess.CalledProcessError as e:
        tail = (e.stderr or "").strip().splitlines()
        return False, round(time.time() - t0, 1), (tail[-1][:80] if tail else "failed")
    except subprocess.TimeoutExpired:
        return False, 45.0, "timeout (no server answered)"

tmp = tempfile.mkdtemp()
results = {}
print(f"{'redirector':30s} {'works':>6} {'sec':>6}  note")
print("-" * 78)
for name, prefix in REDIRECTORS.items():
    ok, sec, note = probe(prefix, LFN, tmp)
    results[name] = (ok, prefix)
    print(f"{name:30s} {('YES' if ok else 'NO'):>6} {sec:>6}  {note}")

redirector                      works    sec  note
------------------------------------------------------------------------------
xcache (coffea.casa)               NO    0.0  Run: [FATAL] Invalid address:  (source)
cmseos.fnal.gov (EOS door)        YES    0.0  1118780 bytes


cms-xrd-global (AAA)              YES    2.7  1118780 bytes


## Load through a working redirector

Pick the first redirector that worked and load the file. **`coffea.util.load` does a local `open()` and rejects `root://` URLs** (the key gotcha, identical on LPC and casa), so we `xrdcp` to local scratch first, then load the local copy.

In [3]:
working = [(name, prefix) for name, (ok, prefix) in results.items() if ok]
assert working, "No redirector worked -- see the escape hatch at the end of the notebook."
name, prefix = working[0]
print("loading via:", name)

local = os.path.join(tmp, os.path.basename(LFN))
subprocess.run(["xrdcp", "-f", "-s", prefix + LFN, local], check=True)
output = coffea.util.load(local)
print("loaded:", type(output).__name__, "| n samples:", len(output))
print("first samples:", list(output.keys())[:5])

loading via: cmseos.fnal.gov (EOS door)


loaded: dict | n samples: 179
first samples: ['4Mu_100GeV_0p25GeV_0p02mm', '4Mu_150GeV_0p25GeV_6p7mm', '4Mu_150GeV_0p25GeV_1p3mm', '4Mu_150GeV_0p25GeV_0p13mm', '4Mu_150GeV_0p25GeV_0p013mm']


## The `.meta.yaml` sidecar

Every SIDM `.coffea` ships a `.meta.yaml` sidecar (samples, selections, hist collections, cross sections, the producing commit) — fetched the same way, through the redirector that worked. (Idiomatic alternative: `sidm.tools.metadata`.)

In [4]:
meta_lfn = LFN[:-len(".coffea")] + ".meta.yaml"
meta_local = os.path.join(tmp, os.path.basename(meta_lfn))
subprocess.run(["xrdcp", "-f", "-s", prefix + meta_lfn, meta_local], check=True)
meta = yaml.safe_load(open(meta_local))
print("meta keys:", list(meta.keys()))
for k in ("commit", "selections", "hist_collections", "samples"):
    if k in meta:
        v = meta[k]
        print(f"  {k}:", list(v)[:4] if isinstance(v, (list, dict)) else v)

meta keys: ['created_utc', 'coffea_path', 'n_samples', 'selections', 'hist_collections', 'schema', 'chunksize', 'unweighted_hist', 'sidm_commit', 'coffea_version', 'samples']
  selections: [{'name': 'baseNoLj_noTrigger', 'definition': {'obj_cuts': {'genMus': ['status 1'], 'genEs': ['status 1'], 'electrons': ['pT > 10 GeV', '|eta| < 2.4', 'MVANonIsoWPL'], 'muons': ['looseID', 'pT > 5 GeV', '|eta| < 2.4'], 'photons': ['pT > 20 GeV', '|eta| < 2.5', 'Custom Cutbased', 'pixelSeed', 'Photon DR Veto 0p025'], 'dsaMuons': ['pT > 10 GeV', '|eta| < 2.4', 'displaced ID', 'all']}, 'evt_cuts': [['PV filter']]}}]
  hist_collections: [{'name': 'genA_lifetime', 'hists': ['genAs_n', 'genAs_pt', 'genAs_gamma', 'genAs_betagamma', 'genAs_lxy', 'genAs_lxyz', 'genAs_lxyz_proper', 'genAs_toMu_lxy', 'genAs_toMu_lxyz', 'genAs_toMu_lxyz_proper', 'genAs_toE_lxy', 'genAs_toE_lxyz', 'genAs_toE_lxyz_proper']}]
  samples: [{'name': '4Mu_100GeV_0p25GeV_0p02mm', 'n_files': 1, 'xsec_pb': 0.001, 'metadata': {'skim_factor

## Inspect the accumulator

`output[sample]` carries `hists` (boost-histogram objects) and a `cutflow`. Shown defensively so it works regardless of the exact layout.

In [5]:
sample = list(output.keys())[0]
acc = output[sample]
print("sample:", sample)
print("keys:", list(acc.keys()) if hasattr(acc, "keys") else type(acc).__name__)
if hasattr(acc, "keys"):
    if "hists" in acc:
        print("  hists:", list(acc["hists"].keys())[:8])
    if "cutflow" in acc:
        cf = acc["cutflow"]
        print("  cutflow:", dict(list(cf.items())[:5]) if hasattr(cf, "items") else cf)

sample: 4Mu_100GeV_0p25GeV_0p02mm
keys: ['cutflow', 'hists', 'counters', 'metadata']
  hists: ['genAs_n', 'genAs_pt', 'genAs_gamma', 'genAs_betagamma', 'genAs_lxy', 'genAs_lxyz', 'genAs_lxyz_proper', 'genAs_toMu_lxy']
  cutflow: {'baseNoLj_noTrigger': <sidm.tools.cutflow.SimpleCutflow object at 0x7f76380c7550>}


## If nothing works on coffea.casa

If the probe table shows **all** redirectors failing on casa (xcache can't resolve the
non-DAS-catalogued group area **and** the direct `cmseos` door has no credential in the session),
the portable escape hatch is to move the bytes without relying on casa's credential:

- from a host that holds a `cms` proxy (e.g. LPC after `voms-proxy-init --voms cms`), `xrdcp` the
  file somewhere casa can reach, **or**
- **upload the `.coffea` into the coffea.casa session** (it's ~1 MB here) and `coffea.util.load`
  the local copy.

Either way the load step is identical — only how the local copy arrives changes. (For the lpcmetx
outputs the confirmed answer is simpler: use `xcache` on coffea.casa, per the table at the top.)